# Tarea — RAWG API Explorer

**API:** [RAWG Video Games Database](https://rawg.io/apidocs)  
**Base URL:** `https://api.rawg.io/api`

In [ ]:
import requests

API_KEY = "dfee4a4db65a44d29d7aca293922ae12"
BASE_URL = "https://api.rawg.io/api"

class RAWGClient:
    """Cliente simple para la RAWG API con conteo de requests."""

    def __init__(self, api_key: str):
        self.api_key = api_key
        self._requests_count = 0

    def get(self, endpoint: str, params: dict = None) -> dict:
        """Realiza un GET a BASE_URL/endpoint y devuelve el JSON."""
        url = f"{BASE_URL}/{endpoint.lstrip('/')}"
        params = params or {}
        params["key"] = self.api_key
        response = requests.get(url, params=params)
        response.raise_for_status()
        self._requests_count += 1
        return response.json()

    def resumen_requests(self) -> str:
        return f"Total de requests realizados: {self._requests_count}"

client = RAWGClient(API_KEY)
print("Cliente RAWG inicializado correctamente.")

---
## Parte A — Exploración General *(2 pts)*

### A1 — *(1 pt)*
¿Cuántos juegos tiene registrados RAWG en total?  
Imprimir el número con un mensaje claro.  
*Hint: el campo `count` está en la respuesta del endpoint `/games`.*

In [ ]:
# A1 — Total de juegos registrados en RAWG
data = client.get("/games")
total_games = data["count"]
print(f"RAWG tiene registrados {total_games:,} juegos en total.")

---
## Parte B — Análisis por Categoría *(2 pts)*

### B1 — *(1 pt)*
¿Cuáles son los **top 5 juegos mejor valorados** de todos los tiempos según Metacritic?  
Mostrar: nombre, rating y puntaje Metacritic.

In [ ]:
# B1 — Top 5 juegos mejor valorados según Metacritic
data_b1 = client.get("/games", params={"ordering": "-metacritic", "page_size": 5})

print("Top 5 juegos mejor valorados según Metacritic:\n")
print(f"{'#':<4} {'Nombre':<45} {'Rating':>7} {'Metacritic':>11}")
print("-" * 70)
for i, game in enumerate(data_b1["results"], start=1):
    print(f"{i:<4} {game['name']:<45} {game['rating']:>7.2f} {game['metacritic']:>11}")

### B2 — *(1 pt)*
¿Cuáles son los **10 mejores juegos disponibles en Steam** (`store_id=1`)?  
Mostrar: nombre, rating y puntaje Metacritic.

In [ ]:
# B2 — Top 10 juegos disponibles en Steam (store_id=1)
data_b2 = client.get("/games", params={"stores": 1, "ordering": "-metacritic", "page_size": 10})

print("Top 10 juegos disponibles en Steam según Metacritic:\n")
print(f"{'#':<4} {'Nombre':<45} {'Rating':>7} {'Metacritic':>11}")
print("-" * 70)
for i, game in enumerate(data_b2["results"], start=1):
    print(f"{i:<4} {game['name']:<45} {game['rating']:>7.2f} {game['metacritic']:>11}")

---
## Parte C — Comparaciones *(3 pts)*

### C1 — *(0.5 pts)*
Comparar los **top 5 juegos en PC** (`platform_id=4`) vs **top 5 en PS5** (`platform_id=187`).  
¿Qué plataforma tiene los juegos mejor valorados?

In [ ]:
# C1 — PC vs PS5: top 5 por Metacritic
def get_top5(platform_id, platform_name):
    data = client.get("/games", params={
        "platforms": platform_id,
        "ordering": "-metacritic",
        "page_size": 5
    })
    games = data["results"]
    print(f"\nTop 5 en {platform_name}:")
    print(f"{'#':<4} {'Nombre':<45} {'Rating':>7} {'Metacritic':>11}")
    print("-" * 70)
    for i, g in enumerate(games, 1):
        print(f"{i:<4} {g['name']:<45} {g['rating']:>7.2f} {g['metacritic']:>11}")
    avg_metacritic = sum(g["metacritic"] for g in games) / len(games)
    avg_rating     = sum(g["rating"]     for g in games) / len(games)
    return games, avg_metacritic, avg_rating

pc_games,  pc_meta,  pc_rat  = get_top5(4,   "PC")
ps5_games, ps5_meta, ps5_rat = get_top5(187, "PS5")

print("\n--- Comparación de promedios ---")
print(f"{'Plataforma':<10} {'Avg Metacritic':>15} {'Avg Rating':>12}")
print("-" * 40)
print(f"{'PC':<10} {pc_meta:>15.2f} {pc_rat:>12.2f}")
print(f"{'PS5':<10} {ps5_meta:>15.2f} {ps5_rat:>12.2f}")

winner_meta = "PC" if pc_meta > ps5_meta else "PS5"
winner_rat  = "PC" if pc_rat  > ps5_rat  else "PS5"
print(f"\nPlataforma con mayor Metacritic promedio: {winner_meta}")
print(f"Plataforma con mayor Rating promedio:     {winner_rat}")

### C2 — *(0.5 pts)*
Elegir **3 juegos famosos** y construir una tabla comparativa con:  
nombre, rating, metacritic, géneros y plataformas.

In [ ]:
# C2 — Tabla comparativa de 3 juegos famosos
famous_games = ["The Witcher 3: Wild Hunt", "Red Dead Redemption 2", "Elden Ring"]

rows = []
for title in famous_games:
    result = client.get("/games", params={"search": title, "page_size": 1})
    g = result["results"][0]
    genres    = ", ".join(gen["name"] for gen in g.get("genres", []))
    platforms = ", ".join(p["platform"]["name"] for p in g.get("platforms", []))
    rows.append({
        "Nombre":     g["name"],
        "Rating":     g["rating"],
        "Metacritic": g["metacritic"],
        "Géneros":    genres,
        "Plataformas": platforms,
    })

# Mostrar tabla
print(f"{'Nombre':<35} {'Rating':>7} {'Meta':>6}  {'Géneros':<30} {'Plataformas'}")
print("-" * 110)
for r in rows:
    print(f"{r['Nombre']:<35} {r['Rating']:>7.2f} {r['Metacritic']:>6}  {r['Géneros']:<30} {r['Plataformas']}")

### C3 — *(0.5 pts)*
Consultar el top 5 de juegos de al menos **4 géneros distintos**, calcular el **rating promedio** de cada uno  
y determinar qué género produce los mejores juegos según los usuarios.

In [ ]:
# C3 — Rating promedio del top 5 por género
genres_to_analyze = {
    "Action":    4,
    "RPG":       5,
    "Shooter":   2,
    "Adventure": 3,
    "Strategy":  10,
}

genre_avgs = {}
for genre_name, genre_id in genres_to_analyze.items():
    data = client.get("/games", params={
        "genres": genre_id,
        "ordering": "-rating",
        "page_size": 5
    })
    games = data["results"]
    avg = sum(g["rating"] for g in games) / len(games)
    genre_avgs[genre_name] = avg
    print(f"{genre_name:<12} → avg rating: {avg:.3f}  |  juegos: {', '.join(g['name'] for g in games)}")

best_genre = max(genre_avgs, key=genre_avgs.get)
print(f"\nGénero con mejor rating promedio: {best_genre} ({genre_avgs[best_genre]:.3f})")

### C4 — *(0.5 pts)*
Comparar los mejores juegos de **3 años distintos**.  
¿En qué año se lanzaron los juegos con mayor Metacritic promedio?

In [ ]:
# C4 — Comparación de mejores juegos por año
years = [2015, 2019, 2022]

year_avgs = {}
for year in years:
    data = client.get("/games", params={
        "dates": f"{year}-01-01,{year}-12-31",
        "ordering": "-metacritic",
        "page_size": 5
    })
    games = data["results"]
    avg_meta = sum(g["metacritic"] for g in games if g["metacritic"]) / len(games)
    year_avgs[year] = avg_meta
    print(f"\nAño {year} — Avg Metacritic: {avg_meta:.1f}")
    print(f"  {'Nombre':<45} {'Metacritic':>11}")
    print("  " + "-" * 58)
    for g in games:
        print(f"  {g['name']:<45} {g['metacritic']:>11}")

best_year = max(year_avgs, key=year_avgs.get)
print(f"\nAño con mayor Metacritic promedio: {best_year} ({year_avgs[best_year]:.1f})")

### C5 — *(1.0 pt)*
Exportar los **top 20 juegos de todos los tiempos** a un CSV llamado `top20_rawg.csv` dentro de `api/output/`.  
Columnas requeridas: `name`, `rating`, `metacritic`, `release_date`, `main_genre`.  
Mostrar las primeras 5 filas del archivo generado.

In [ ]:
import csv
import os

# C5 — Exportar top 20 a CSV
data_c5 = client.get("/games", params={"ordering": "-metacritic", "page_size": 20})
games_c5 = data_c5["results"]

output_path = os.path.join(os.path.dirname(os.path.abspath("__file__")), "output", "top20_rawg.csv")
os.makedirs(os.path.dirname(output_path), exist_ok=True)

fieldnames = ["name", "rating", "metacritic", "release_date", "main_genre"]

with open(output_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    for g in games_c5:
        main_genre = g["genres"][0]["name"] if g.get("genres") else "N/A"
        writer.writerow({
            "name":         g["name"],
            "rating":       g["rating"],
            "metacritic":   g["metacritic"],
            "release_date": g.get("released", "N/A"),
            "main_genre":   main_genre,
        })

print(f"CSV exportado a: {output_path}\n")

# Mostrar primeras 5 filas
with open(output_path, encoding="utf-8") as f:
    reader = csv.DictReader(f)
    rows_preview = [row for _, row in zip(range(5), reader)]

print(f"{'name':<45} {'rating':>7} {'metacritic':>11} {'release_date':>13} {'main_genre'}")
print("-" * 95)
for row in rows_preview:
    print(f"{row['name']:<45} {row['rating']:>7} {row['metacritic']:>11} {row['release_date']:>13} {row['main_genre']}")